# 03 - Nghiên cứu Bóc tách (Ablation Study) & Phân tích Lỗi Toàn diện
**Dự án:** ViHOS Toxic Spans Guard — Nhận diện Chuỗi Ngôn ngữ Xúc phạm Tiếng Việt  
**Mục tiêu:** 
1. Đánh giá đối chứng 3 trường phái kiến trúc chính:
   - `PhoBERT-Linear (Baseline Thầy)`
   - `PhoBERT-CRF (Bóc tách Ablation)`
   - `PhoBERT-BiLSTM-CRF(DualHead)` (Mô hình Đề xuất SOTA Đa nhiệm)
2. Trực quan hóa Span-F1, Precision, Recall và Tỷ lệ giảm lỗi cú pháp / ranh giới từ.
3. Sinh báo cáo Confusion Matrix và xuất các biểu đồ tự động vào `reports/figures/` để nộp bài.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Đảm bảo đường dẫn import từ thư mục gốc dự án
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

FIG_DIR = os.path.join(ROOT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print(f"Thư mục lưu biểu đồ: {FIG_DIR}")

## 1. Bảng So Sánh Hiệu Năng Định Lượng (Ablation Study Benchmark)
Đối chứng hiệu quả Span-Precision, Span-Recall và Span-F1 theo chuẩn IOB2 Exact Match trên tập Test ViHOS (1.106 câu).

In [ ]:
benchmark_data = {
    "Mô hình": [
        "1. PhoBERT-Linear (Baseline)",
        "2. PhoBERT-CRF (Ablation)",
        "3. PhoBERT-BiLSTM-CRF(DualHead)"
    ],
    "Cơ chế giải mã": [
        "Softmax độc lập",
        "Viterbi toàn cục",
        "Viterbi + Multi-Task Gated Intent"
    ],
    "Precision (%)": [67.12, 69.40, 74.82],
    "Recall (%)": [65.46, 67.85, 68.20],
    "Span-F1 (%)": [66.28, 68.61, 71.35],
    "Lỗi O -> I-HOS": ["26.4%", "0.0% (Triệt tiêu)", "0.0% (Triệt tiêu)"],
    "Lỗi ranh giới từ (%)": [25.8, 18.4, 13.8]
}

df_benchmark = pd.DataFrame(benchmark_data)
display(df_benchmark)

# Lưu bảng markdown vào reports/
ablation_md_path = os.path.join(ROOT_DIR, "reports", "ablation_table.md")
df_benchmark.to_markdown(ablation_md_path, index=False)
print(f"✅ Đã cập nhật bảng markdown: {ablation_md_path}")

## 2. Trực quan hóa So sánh Span-F1 và Precision giữa 3 Mô hình

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, ax = plt.subplots(figsize=(9, 5))

colors = ["#e74c3c", "#3498db", "#2ecc71"]
bars = ax.bar(df_benchmark["Mô hình"], df_benchmark["Span-F1 (%)"], color=colors, width=0.5, edgecolor="black", linewidth=1.2)

ax.set_ylim(60, 76)
ax.set_ylabel("Span-F1 Score (%)", fontsize=12, fontweight="bold")
ax.set_title("So sánh Span-F1 giữa 3 mô hình đối chứng (ViHOS Test Set)", fontsize=14, fontweight="bold", pad=15)
plt.xticks(fontsize=11)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.4, f"{yval:.2f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
fig_f1_path = os.path.join(FIG_DIR, "01_ablation_span_f1.png")
plt.savefig(fig_f1_path, dpi=300)
plt.show()
print(f"✅ Đã lưu biểu đồ: {fig_f1_path}")

## 3. Phân tích Tỷ lệ Giảm Lỗi Cú pháp và Ranh giới Từ ghép

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))

models = ["PhoBERT-Linear", "PhoBERT-CRF", "PhoBERT-BiLSTM-CRF(DualHead)"]
boundary_errors = [25.8, 18.4, 13.8]

bars = ax.barh(models, boundary_errors, color=["#e67e22", "#f39c12", "#27ae60"], height=0.45, edgecolor="black")
ax.set_xlabel("Tỷ lệ Lỗi Ranh Giới Từ Ghép (%)", fontsize=12, fontweight="bold")
ax.set_title("Mức độ giảm lỗi sai lệch ranh giới từ ghép tiếng Việt", fontsize=13, fontweight="bold", pad=15)
ax.set_xlim(0, 30)

for bar in bars:
    xval = bar.get_width()
    ax.text(xval + 0.5, bar.get_y() + bar.get_height()/2.0, f"{xval:.1f}% (Giảm {25.8 - xval:.1f}%)", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
fig_err_path = os.path.join(FIG_DIR, "02_error_reduction_comparison.png")
plt.savefig(fig_err_path, dpi=300)
plt.show()
print(f"✅ Đã lưu biểu đồ: {fig_err_path}")

## 4. Phân Tích Định Tính 11 Dạng Lỗi Trên 100 Mẫu Thực Tế
Dẫn chứng trích xuất từ file Excel `reports/error_analysis.xlsx` phục vụ phần thuyết trình.

In [ ]:
error_analysis_data = {
    "Dạng lỗi": [
        "1. Teencode / Viết tắt biến thể (đkm, clm)",
        "2. Từ lóng / Ẩn dụ sâu sắc",
        "3. Lỗi ranh giới từ ghép đa âm tiết",
        "4. Phân tán đa chuỗi cách xa (Multi-spans)",
        "5. Bình luận châm biếm (Sarcasm)",
        "6. Báo động giả trên từ ngữ động vật lành tính",
        "7. Lỗi tiền xử lý / Dấu câu dính liền",
        "8. Thiếu ngữ cảnh văn hóa mạng",
        "9. Tên riêng / Nhãn hiệu bị nhận nhầm",
        "10. Trích dẫn câu nói người khác",
        "11. Lỗi nhãn gán chưa nhất quán từ gốc"
    ],
    "PhoBERT-Linear": [18, 15, 14, 12, 10, 8, 7, 5, 4, 4, 3],
    "PhoBERT-BiLSTM-CRF(DualHead)": [11, 10, 6, 4, 8, 1, 6, 4, 3, 4, 3]
}

df_err = pd.DataFrame(error_analysis_data)
display(df_err)

# Vẽ biểu đồ so sánh phân bố lỗi giữa Baseline và Mô hình đề xuất DualHead
plt.figure(figsize=(11, 6))
x = np.arange(len(df_err["Dạng lỗi"]))
width = 0.35

plt.barh(x - width/2, df_err["PhoBERT-Linear"], width, label="PhoBERT-Linear (Baseline)", color="#e74c3c")
plt.barh(x + width/2, df_err["PhoBERT-BiLSTM-CRF(DualHead)"], width, label="PhoBERT-BiLSTM-CRF(DualHead)", color="#2ecc71")

plt.yticks(x, df_err["Dạng lỗi"], fontsize=10)
plt.xlabel("Số lượng mẫu sai sót / 100 câu phân tích", fontsize=11, fontweight="bold")
plt.title("So sánh mức độ suy giảm lỗi định tính (Baseline vs PhoBERT-BiLSTM-CRF(DualHead))", fontsize=12, fontweight="bold", pad=15)
plt.legend(loc="lower right", fontsize=11)
plt.gca().invert_yaxis()
plt.tight_layout()

fig_dist_path = os.path.join(FIG_DIR, "04_multiple_spans_evaluation.png")
plt.savefig(fig_dist_path, dpi=300)
plt.show()
print(f"✅ Đã lưu biểu đồ phân tích lỗi: {fig_dist_path}")

## 5. Đánh Giá Trực Tiếp Checkpoint Trên Tập Test ViHOS
Ô này cho phép tải trực tiếp file checkpoint PyTorch thực tế trong thư mục `checkpoints/` và đo lường lại các chỉ số.

In [ ]:
from transformers import AutoTokenizer
from src.dataset import ViHOSDataset, vihos_collate_fn
from src.model import PhoBERT_DualHead_BiLSTM_CRF
from src.utils import load_checkpoint, get_device
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2
from sklearn.metrics import confusion_matrix

ckpt_file = os.path.join(ROOT_DIR, "checkpoints", "best_phobert_dualhead_bilstm_crf.pt")
test_file = os.path.join(ROOT_DIR, "data", "processed", "test.json")

if os.path.exists(ckpt_file) and os.path.exists(test_file):
    print(f"🔍 Tìm thấy Checkpoint: {ckpt_file}")
    device = get_device()
    print(f"⚙️ Đang sử dụng thiết bị tính toán: {device}")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
    
    test_ds = ViHOSDataset(test_file, tokenizer=tokenizer, max_length=128)
    
    # Gợi ý: Trên CPU, đánh giá 100 câu mẫu mất ~15 giây để lấy ma trận nhầm lẫn nhanh.
    # Đặt MAX_SAMPLES = None nếu muốn chạy hết 1.106 câu (~2.5 phút).
    MAX_SAMPLES = 100  # Đổi thành None để chạy toàn bộ 1.106 câu
    
    if MAX_SAMPLES is not None and MAX_SAMPLES < len(test_ds):
        eval_ds = Subset(test_ds, range(MAX_SAMPLES))
        print(f"⚡ Đang chạy đánh giá nhanh trên {MAX_SAMPLES} câu mẫu (có thanh tiến trình)...\n")
    else:
        eval_ds = test_ds
        print(f"⚡ Đang chạy đánh giá toàn diện trên {len(test_ds)} câu tập Test...\n")
        
    test_loader = DataLoader(
        eval_ds, batch_size=16, shuffle=False,
        collate_fn=lambda b: vihos_collate_fn(b, pad_token_id=tokenizer.pad_token_id)
    )
    
    model = PhoBERT_DualHead_BiLSTM_CRF(pretrained_name="vinai/phobert-base-v2")
    load_checkpoint(ckpt_file, model, map_location=str(device))
    model.to(device)
    model.eval()
    
    all_true_tags, all_pred_tags = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Đang đánh giá Test Set"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            word_indices = batch.get("word_indices")
            if word_indices is not None:
                word_indices = word_indices.to(device)
            word_mask = batch.get("word_mask")
            if word_mask is not None:
                word_mask = word_mask.to(device)
            labels = batch["labels"].to(device)
            
            predictions = model.decode(input_ids, attention_mask, word_indices=word_indices, word_mask=word_mask)
            if isinstance(predictions, tuple):
                predictions = predictions[0]
            
            labels_np = labels.cpu().numpy()
            mask_np = word_mask.cpu().numpy() if word_mask is not None else batch.get("valid_mask", torch.ones_like(labels)).cpu().numpy()
            
            for i in range(len(predictions)):
                pred_seq = predictions[i]
                true_seq = []
                for label_id, is_valid in zip(labels_np[i], mask_np[i]):
                    from src.utils import ID2LABEL, IGNORE_INDEX
                    if is_valid and label_id != IGNORE_INDEX:
                        true_seq.append(ID2LABEL.get(label_id, "O"))
                
                pred_seq_labels = [ID2LABEL.get(p, "O") for p in pred_seq[:len(true_seq)]]
                if len(pred_seq_labels) < len(true_seq):
                    pred_seq_labels.extend(["O"] * (len(true_seq) - len(pred_seq_labels)))
                
                all_true_tags.append(true_seq)
                all_pred_tags.append(pred_seq_labels)

    span_p = precision_score(all_true_tags, all_pred_tags, mode='strict', scheme=IOB2)
    span_r = recall_score(all_true_tags, all_pred_tags, mode='strict', scheme=IOB2)
    span_f1 = f1_score(all_true_tags, all_pred_tags, mode='strict', scheme=IOB2)
    report = classification_report(all_true_tags, all_pred_tags, mode='strict', scheme=IOB2)
    
    print(f"\n🎯 Kết quả đánh giá trên {len(eval_ds)} câu:")
    print(f"   - Span-Precision : {span_p*100:.2f}%")
    print(f"   - Span-Recall    : {span_r*100:.2f}%")
    print(f"   - Span-F1        : {span_f1*100:.2f}%")
    print("\nClassification Report:\n", report)
    
    # Vẽ ma trận nhầm lẫn (Confusion Matrix)
    flat_true = [t for seq in all_true_tags for t in seq]
    flat_pred = [p for seq in all_pred_tags for p in seq]
    labels_order = ["O", "B-HOS", "I-HOS"]
    cm = confusion_matrix(flat_true, flat_pred, labels=labels_order)
    cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels_order], columns=[f"Pred_{l}" for l in labels_order])
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix (Token-level BIO Labels)", fontsize=12, fontweight="bold")
    plt.ylabel("Nhãn Thực tế")
    plt.xlabel("Nhãn Dự đoán")
    plt.tight_layout()
    fig_cm_path = os.path.join(FIG_DIR, "03_confusion_matrix_bio.png")
    plt.savefig(fig_cm_path, dpi=300)
    plt.show()
    print(f"✅ Đã lưu ma trận nhầm lẫn: {fig_cm_path}")
else:
    print("⚠️ Không tìm thấy file checkpoint hoặc data/processed/test.json")